# Data Cleaning Script — Q-Commerce Raw Files

In [584]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [585]:
INPUT_DIR  = "Existing-Files"
OUTPUT_DIR = "Output-files"
sku_mapping = pd.read_csv("/Users/akshaybisht/Desktop/E-Automation/Existing-Files/SKU Mapping.csv")

## SKU MAPPING CLEANING

In [586]:
sku_mapping.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Sumo ID               17 non-null     float64
 1   Category              24 non-null     object 
 2   SKU                   24 non-null     object 
 3   MRP                   17 non-null     float64
 4   Blinkit Identifier    22 non-null     float64
 5   Status                17 non-null     object 
 6   Instamart Identifier  21 non-null     float64
 7   Status.1              17 non-null     object 
 8   Zepto Identifier      21 non-null     object 
 9   Status.2              17 non-null     object 
 10  FKM Identifier        10 non-null     object 
 11  Status.3              17 non-null     object 
 12  BB Identifier         16 non-null     float64
 13  Status.4              17 non-null     object 
dtypes: float64(5), object(9)
memory usage: 2.8+ KB


In [587]:
for obj_col in sku_mapping.select_dtypes(include = "object"):
    sku_mapping[obj_col] = sku_mapping[obj_col].astype("string").str.strip()
sku_mapping.drop(columns = ["MRP","Status","Status.1","Status.2","Status.3","Status.4"], inplace=True)
sku_mapping.rename(columns={"Blinkit Identifier":"blinkit_identifier",
                            "Instamart Identifier":"instamart_identifier",
                            "Zepto Identifier":"zepto_identifier",
                            "FKM Identifier":"FKM_identifier",
                            "BB Identifier":"BB_identifier",
                            "Sumo ID":"internal_id",
                            "SKU":"internal_name"}, inplace=True)

## 1. Blinkit

### Prepping the Dump

In [588]:
df_blinkit = pd.read_csv(f"{INPUT_DIR}/blinkit-raw.csv", low_memory=False)

df_blinkit.drop(columns=["Internal SKU"], inplace=True)
df_blinkit = df_blinkit[["item_id", "item_name", "manufacturer_id", "manufacturer_name",
                          "city_id", "city_name", "category", "date", "qty_sold", "mrp"]]
df_blinkit.drop(index=220035, inplace=True)

for col in df_blinkit.select_dtypes(include="object"):
    df_blinkit[col] = df_blinkit[col].str.strip().astype("string")

df_blinkit["item_id"]         = pd.to_numeric(df_blinkit["item_id"])
df_blinkit["manufacturer_id"] = pd.to_numeric(df_blinkit["manufacturer_id"])
df_blinkit["city_id"]         = pd.to_numeric(df_blinkit["city_id"])
df_blinkit["qty_sold"]        = pd.to_numeric(df_blinkit["qty_sold"])
df_blinkit["mrp"]             = pd.to_numeric(df_blinkit["mrp"])
df_blinkit["date"]            = pd.to_datetime(df_blinkit["date"], dayfirst=True)

df_blinkit.drop_duplicates(inplace=True)
df_blinkit.sort_values("date", inplace=True)

In [589]:
df_blinkit.rename(columns={"item_id":"blinkit_identifier",
                           "item_name":"blinkit_name",
                           "item_id":"blinkit_identifier"}, inplace=True)

In [590]:
df_blinkit.info()
df_blinkit.head()

<class 'pandas.core.frame.DataFrame'>
Index: 230452 entries, 0 to 230452
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   blinkit_identifier  230452 non-null  Int64         
 1   blinkit_name        230452 non-null  string        
 2   manufacturer_id     230452 non-null  Int64         
 3   manufacturer_name   230452 non-null  string        
 4   city_id             230452 non-null  Int64         
 5   city_name           230452 non-null  string        
 6   category            230452 non-null  string        
 7   date                230452 non-null  datetime64[ns]
 8   qty_sold            230452 non-null  Int64         
 9   mrp                 230452 non-null  Int64         
dtypes: Int64(5), datetime64[ns](1), string(4)
memory usage: 20.4 MB


,blinkit_identifier,blinkit_name,manufacturer_id,manufacturer_name,city_id,city_name,category,date,qty_sold,mrp
0,10174324,Chaayos Festive Tea Gift Box,1546,Sunshine Teahouse Pvt. Ltd.,2,Mumbai,Cold Drinks & Juices,2024-12-01,3,1647
122,10074977,Chaayos Adrak Elaichi Tea(Pouch),1546,Sunshine Teahouse Pvt. Ltd.,125,Panchkula,"Tea, Coffee & Milk Drinks",2024-12-01,4,556
123,10074977,Chaayos Adrak Elaichi Tea(Pouch),1546,Sunshine Teahouse Pvt. Ltd.,17,Bhopal,"Tea, Coffee & Milk Drinks",2024-12-01,1,139
124,10074977,Chaayos Adrak Elaichi Tea(Pouch),1546,Sunshine Teahouse Pvt. Ltd.,91,Bahadurgarh,"Tea, Coffee & Milk Drinks",2024-12-01,1,139
125,10074977,Chaayos Adrak Elaichi Tea(Pouch),1546,Sunshine Teahouse Pvt. Ltd.,178,Karnal,"Tea, Coffee & Milk Drinks",2024-12-01,1,139


### Prepping SKU MAPPING to MERGE with DUMP

In [591]:
sku_mapping.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   internal_id           17 non-null     float64
 1   Category              24 non-null     string 
 2   internal_name         24 non-null     string 
 3   blinkit_identifier    22 non-null     float64
 4   instamart_identifier  21 non-null     float64
 5   zepto_identifier      21 non-null     string 
 6   FKM_identifier        10 non-null     string 
 7   BB_identifier         16 non-null     float64
dtypes: float64(4), string(4)
memory usage: 1.6 KB


In [592]:
sku_mapping_blinkit = sku_mapping[["blinkit_identifier","internal_id","internal_name","Category"]].copy()
sku_mapping_blinkit.head()

,blinkit_identifier,internal_id,internal_name,Category
0,10074977.0,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
1,10191319.0,5751.0,Chaayos Adrak Elaichi 250g,Chai Patti
2,10256646.0,6330.0,Chaayos Adrak Elaichi 500g,Chai Patti
3,10074976.0,1504.0,Chaayos Masala Tea 100g,Chai Patti
4,10191317.0,5749.0,Chaayos Masala Tea 250g,Chai Patti


In [593]:
blinkit_base = df_blinkit.merge(sku_mapping_blinkit, on="blinkit_identifier", how="left")
blinkit_base.head()

,blinkit_identifier,blinkit_name,manufacturer_id,manufacturer_name,city_id,city_name,category,date,qty_sold,mrp,internal_id,internal_name,Category
0,10174324,Chaayos Festive Tea Gift Box,1546,Sunshine Teahouse Pvt. Ltd.,2,Mumbai,Cold Drinks & Juices,2024-12-01,3,1647,NaN,Discontinue,Discontinue
1,10074977,Chaayos Adrak Elaichi Tea(Pouch),1546,Sunshine Teahouse Pvt. Ltd.,125,Panchkula,"Tea, Coffee & Milk Drinks",2024-12-01,4,556,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
2,10074977,Chaayos Adrak Elaichi Tea(Pouch),1546,Sunshine Teahouse Pvt. Ltd.,17,Bhopal,"Tea, Coffee & Milk Drinks",2024-12-01,1,139,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
3,10074977,Chaayos Adrak Elaichi Tea(Pouch),1546,Sunshine Teahouse Pvt. Ltd.,91,Bahadurgarh,"Tea, Coffee & Milk Drinks",2024-12-01,1,139,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
4,10074977,Chaayos Adrak Elaichi Tea(Pouch),1546,Sunshine Teahouse Pvt. Ltd.,178,Karnal,"Tea, Coffee & Milk Drinks",2024-12-01,1,139,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti


## 2. Zepto

In [594]:
df_zepto = pd.read_csv(f"{INPUT_DIR}/zepto-raw.csv", low_memory=False)

df_zepto.drop(df_zepto.iloc[:, 0:10], axis=1, inplace=True)
df_zepto.rename(columns={
    "Date.1"           : "date",
    "SKU Name"         : "zepto_name",
    "SKU ID"           : "zepto_identifier",
    "City.1"           : "city",
    "Brand Name"       : "brand_name",
    "Manufacturer ID"  : "manufacture_id",
    "Manufacturer Name": "manufacture_name",
    "SKU Category"     : "sku_category",
    "Quantity"         : "quantity",
    "GMV.1"            : "gmv",
}, inplace=True)

df_zepto["date"] = pd.to_datetime(df_zepto["date"].str.replace("-", "/"), dayfirst=True)

for col in df_zepto.select_dtypes(include="object"):
    df_zepto[col] = df_zepto[col].str.strip().astype("string")

df_zepto.dropna(subset=["quantity"], inplace=True)
df_zepto.sort_values("date", inplace=True)

In [595]:
df_zepto.info()
df_zepto.head()

<class 'pandas.core.frame.DataFrame'>
Index: 34691 entries, 0 to 35580
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              34691 non-null  datetime64[ns]
 1   zepto_name        34691 non-null  string        
 2   zepto_identifier  34691 non-null  string        
 3   city              34691 non-null  string        
 4   brand_name        34691 non-null  string        
 5   manufacture_id    34691 non-null  string        
 6   manufacture_name  34691 non-null  string        
 7   sku_category      34691 non-null  string        
 8   quantity          34691 non-null  float64       
 9   gmv               34691 non-null  float64       
dtypes: datetime64[ns](1), float64(2), string(7)
memory usage: 2.9 MB


,date,zepto_name,zepto_identifier,city,brand_name,manufacture_id,manufacture_name,sku_category,quantity,gmv
0,2025-10-01,Chaayos Adrak Elachi Instant Tea Premix - Low ...,102842f3-8ded-4069-a4d0-1448fc74f924,Chennai,Chaayos,ec4e6e5a-d9fe-4edf-8d7c-52d2fa3c8817,Chaayos,"Tea, Coffee & More",3.0,627.0
104,2025-10-01,Chaayos Ginger Cardamom Tea 100.0 GRAM,a7e1f97f-08a3-4543-a82e-ff96af10a513,Kolkata,Chaayos,ec4e6e5a-d9fe-4edf-8d7c-52d2fa3c8817,Chaayos,"Tea, Coffee & More",5.0,695.0
105,2025-10-01,Chaayos Ginger Cardamom Tea 100.0 GRAM,a7e1f97f-08a3-4543-a82e-ff96af10a513,Puducherry,Chaayos,ec4e6e5a-d9fe-4edf-8d7c-52d2fa3c8817,Chaayos,"Tea, Coffee & More",1.0,139.0
106,2025-10-01,Chaayos Ginger Cardamom Tea 100.0 GRAM,a7e1f97f-08a3-4543-a82e-ff96af10a513,Vijayawada,Chaayos,ec4e6e5a-d9fe-4edf-8d7c-52d2fa3c8817,Chaayos,"Tea, Coffee & More",1.0,139.0
107,2025-10-01,Chaayos Ginger Cardamom Tea 100.0 GRAM,a7e1f97f-08a3-4543-a82e-ff96af10a513,Faridabad,Chaayos,ec4e6e5a-d9fe-4edf-8d7c-52d2fa3c8817,Chaayos,"Tea, Coffee & More",1.0,139.0


### Prepping SKU MAPPING to MERGE with DUMP

In [596]:
sku_mapping.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   internal_id           17 non-null     float64
 1   Category              24 non-null     string 
 2   internal_name         24 non-null     string 
 3   blinkit_identifier    22 non-null     float64
 4   instamart_identifier  21 non-null     float64
 5   zepto_identifier      21 non-null     string 
 6   FKM_identifier        10 non-null     string 
 7   BB_identifier         16 non-null     float64
dtypes: float64(4), string(4)
memory usage: 1.6 KB


In [597]:
sku_mapping_zepto = sku_mapping[["zepto_identifier","internal_id","internal_name","Category"]].copy()
sku_mapping_zepto.head()

,zepto_identifier,internal_id,internal_name,Category
0,a7e1f97f-08a3-4543-a82e-ff96af10a513,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
1,d52b7752-4c21-4e72-9958-c7ea98808f5f,5751.0,Chaayos Adrak Elaichi 250g,Chai Patti
2,38fe33c0-f0d0-4a9c-8edb-1ebfdbaee78f,6330.0,Chaayos Adrak Elaichi 500g,Chai Patti
3,bef6de38-e41e-47d8-8062-053f03695e14,1504.0,Chaayos Masala Tea 100g,Chai Patti
4,7e515301-e0b0-457e-ac48-fcf9b2046a44,5749.0,Chaayos Masala Tea 250g,Chai Patti


In [598]:
zepto_base = df_zepto.merge(sku_mapping_zepto, on = "zepto_identifier", how = "left")
zepto_base.head()

,date,zepto_name,zepto_identifier,city,brand_name,manufacture_id,manufacture_name,sku_category,quantity,gmv,internal_id,internal_name,Category
0,2025-10-01,Chaayos Adrak Elachi Instant Tea Premix - Low ...,102842f3-8ded-4069-a4d0-1448fc74f924,Chennai,Chaayos,ec4e6e5a-d9fe-4edf-8d7c-52d2fa3c8817,Chaayos,"Tea, Coffee & More",3.0,627.0,5742.0,Chaayos Adrak Elaichi Premix Pack of 15,Premix
1,2025-10-01,Chaayos Ginger Cardamom Tea 100.0 GRAM,a7e1f97f-08a3-4543-a82e-ff96af10a513,Kolkata,Chaayos,ec4e6e5a-d9fe-4edf-8d7c-52d2fa3c8817,Chaayos,"Tea, Coffee & More",5.0,695.0,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
2,2025-10-01,Chaayos Ginger Cardamom Tea 100.0 GRAM,a7e1f97f-08a3-4543-a82e-ff96af10a513,Puducherry,Chaayos,ec4e6e5a-d9fe-4edf-8d7c-52d2fa3c8817,Chaayos,"Tea, Coffee & More",1.0,139.0,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
3,2025-10-01,Chaayos Ginger Cardamom Tea 100.0 GRAM,a7e1f97f-08a3-4543-a82e-ff96af10a513,Vijayawada,Chaayos,ec4e6e5a-d9fe-4edf-8d7c-52d2fa3c8817,Chaayos,"Tea, Coffee & More",1.0,139.0,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
4,2025-10-01,Chaayos Ginger Cardamom Tea 100.0 GRAM,a7e1f97f-08a3-4543-a82e-ff96af10a513,Faridabad,Chaayos,ec4e6e5a-d9fe-4edf-8d7c-52d2fa3c8817,Chaayos,"Tea, Coffee & More",1.0,139.0,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti


## 3. Instamart

In [599]:
df_instamart = pd.read_csv(f"{INPUT_DIR}/instamart-raw.csv", low_memory=False)

df_instamart.drop(df_instamart.iloc[:, 0:11], axis=1, inplace=True)

for col in df_instamart.select_dtypes(include="object"):
    df_instamart[col] = df_instamart[col].str.strip().astype("string")

df_instamart.rename(columns={"GMV.1": "GMV",
                             "PRODUCT_NAME": "instamart_name",
                             "ITEM_CODE":"instamart_identifier"}, inplace=True)
df_instamart["ORDERED_DATE"] = pd.to_datetime(df_instamart["ORDERED_DATE"], dayfirst=True, format="mixed")
df_instamart.sort_values("ORDERED_DATE", inplace=True)

In [600]:
df_instamart.info()
df_instamart.head()

<class 'pandas.core.frame.DataFrame'>
Index: 82534 entries, 0 to 82533
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   ORDERED_DATE          82534 non-null  datetime64[ns]
 1   CITY                  82534 non-null  string        
 2   AREA_NAME             82534 non-null  string        
 3   STORE_ID              82534 non-null  int64         
 4   L1_CATEGORY           82534 non-null  string        
 5   L2_CATEGORY           82534 non-null  string        
 6   L3_CATEGORY           82534 non-null  string        
 7   instamart_name        82534 non-null  string        
 8   VARIANT               82534 non-null  string        
 9   instamart_identifier  82534 non-null  int64         
 10  COMBO                 82534 non-null  string        
 11  COMBO_ITEM_CODE       4899 non-null   float64       
 12  COMBO_UNITS_SOLD      4899 non-null   float64       
 13  BASE_MRP             

,ORDERED_DATE,CITY,AREA_NAME,STORE_ID,L1_CATEGORY,L2_CATEGORY,L3_CATEGORY,instamart_name,VARIANT,instamart_identifier,COMBO,COMBO_ITEM_CODE,COMBO_UNITS_SOLD,BASE_MRP,UNITS_SOLD,GMV
0,2025-10-01,Agra,civil lines,1401076,"tea, coffee and more",tea,tea premix,chaayos instant tea premix - regular sugar - m...,330 g,399295,Yes,229392.0,1.0,168,1,168
360,2025-10-01,Delhi,vasant kunj,1386536,"tea, coffee and more",tea,tea premix,chaayos masala tea - premium chai patti with 1...,250 g,507889,No,NaN,NaN,299,2,598
359,2025-10-01,Delhi,mayur vihar,1404576,"tea, coffee and more",instant drink mixes,ice tea,chaayos instant peach iced tea premix(box),10 pieces,115595,No,NaN,NaN,119,2,238
358,2025-10-01,Delhi,ashok vihar,816651,"tea, coffee and more",tea,tea premix,chaayos instant tea premix - regular sugar - c...,330 g,200836,No,NaN,NaN,189,3,567
357,2025-10-01,Bhopal,mp nagar,1399345,"tea, coffee and more",tea,tea premix,chaayos instant tea premix - regular sugar - c...,330 g,200836,No,NaN,NaN,189,2,378


### Prepping SKU MAPPING to MERGE with DUMP

In [601]:
sku_mapping_ = sku_mapping[["zepto_identifier","internal_id","internal_name","Category"]].copy()
sku_mapping_zepto.head()

,zepto_identifier,internal_id,internal_name,Category
0,a7e1f97f-08a3-4543-a82e-ff96af10a513,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
1,d52b7752-4c21-4e72-9958-c7ea98808f5f,5751.0,Chaayos Adrak Elaichi 250g,Chai Patti
2,38fe33c0-f0d0-4a9c-8edb-1ebfdbaee78f,6330.0,Chaayos Adrak Elaichi 500g,Chai Patti
3,bef6de38-e41e-47d8-8062-053f03695e14,1504.0,Chaayos Masala Tea 100g,Chai Patti
4,7e515301-e0b0-457e-ac48-fcf9b2046a44,5749.0,Chaayos Masala Tea 250g,Chai Patti


In [602]:
sku_mapping_instamart = sku_mapping[["instamart_identifier","internal_id","internal_name","Category"]].copy()
sku_mapping_instamart.head()

,instamart_identifier,internal_id,internal_name,Category
0,NaN,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
1,635219.0,5751.0,Chaayos Adrak Elaichi 250g,Chai Patti
2,799622.0,6330.0,Chaayos Adrak Elaichi 500g,Chai Patti
3,NaN,1504.0,Chaayos Masala Tea 100g,Chai Patti
4,507889.0,5749.0,Chaayos Masala Tea 250g,Chai Patti


In [603]:
instamart_base = df_instamart.merge(sku_mapping_instamart, on = "instamart_identifier", how = "left")
instamart_base.head()

,ORDERED_DATE,CITY,AREA_NAME,STORE_ID,L1_CATEGORY,L2_CATEGORY,L3_CATEGORY,instamart_name,VARIANT,instamart_identifier,COMBO,COMBO_ITEM_CODE,COMBO_UNITS_SOLD,BASE_MRP,UNITS_SOLD,GMV,internal_id,internal_name,Category
0,2025-10-01,Agra,civil lines,1401076,"tea, coffee and more",tea,tea premix,chaayos instant tea premix - regular sugar - m...,330 g,399295,Yes,229392.0,1.0,168,1,168,4972.0,Chaayos Masala Tea Premix Pack of 15,Premix
1,2025-10-01,Delhi,vasant kunj,1386536,"tea, coffee and more",tea,tea premix,chaayos masala tea - premium chai patti with 1...,250 g,507889,No,NaN,NaN,299,2,598,5749.0,Chaayos Masala Tea 250g,Chai Patti
2,2025-10-01,Delhi,mayur vihar,1404576,"tea, coffee and more",instant drink mixes,ice tea,chaayos instant peach iced tea premix(box),10 pieces,115595,No,NaN,NaN,119,2,238,5915.0,Chaayos Peach Iced Tea Premix,Ice Tea
3,2025-10-01,Delhi,ashok vihar,816651,"tea, coffee and more",tea,tea premix,chaayos instant tea premix - regular sugar - c...,330 g,200836,No,NaN,NaN,189,3,567,4973.0,Chaayos Cardamom Tea Premix Pack of 15,Premix
4,2025-10-01,Bhopal,mp nagar,1399345,"tea, coffee and more",tea,tea premix,chaayos instant tea premix - regular sugar - c...,330 g,200836,No,NaN,NaN,189,2,378,4973.0,Chaayos Cardamom Tea Premix Pack of 15,Premix


## 4. FK Minutes

In [604]:
df_fkminutes = pd.read_csv(f"{INPUT_DIR}/fkminutes-raw.csv", low_memory=False)

df_fkminutes.drop(df_fkminutes.iloc[:, 0:11], axis=1, inplace=True)

for col in df_fkminutes.select_dtypes(include="object"):
    df_fkminutes[col] = df_fkminutes[col].str.strip().astype("string")

# Normalise date string and truncate to date part only
df_fkminutes["order_date_time"] = (
    df_fkminutes["order_date_time"]
    .str.replace("-", "/")
    .str[:10]
)
df_fkminutes["order_date_time"] = pd.to_datetime(df_fkminutes["order_date_time"], dayfirst=True, format="mixed")
df_fkminutes.sort_values("order_date_time", inplace=True)

In [605]:
df_fkminutes.rename(columns={"product_id":"FKM_identifier"}, inplace = True)

In [606]:
df_fkminutes.info()
df_fkminutes.head()

<class 'pandas.core.frame.DataFrame'>
Index: 5958 entries, 0 to 5957
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   order_date_time          5958 non-null   datetime64[ns]
 1   city                     5957 non-null   string        
 2   FKM_identifier           5958 non-null   string        
 3   analytic_business_unit   5958 non-null   string        
 4   analytic_super_category  5958 non-null   string        
 5   analytic_vertical        5958 non-null   string        
 6   brand_csv                5958 non-null   string        
 7   units                    5958 non-null   int64         
 8   mrp                      5958 non-null   int64         
dtypes: datetime64[ns](1), int64(2), string(6)
memory usage: 465.5 KB


,order_date_time,city,FKM_identifier,analytic_business_unit,analytic_super_category,analytic_vertical,brand_csv,units,mrp
0,2025-10-01,Patna,TEAH35UHKB2DCZPV,BGM,FoodAndNutrition,Tea,Chaayos,1,139
22,2025-10-01,Bangalore,TEAH35UP8DYXN3HJ,BGM,FoodAndNutrition,Tea,Chaayos,5,695
21,2025-10-01,Kolkata,TEAH35UHKB2DCZPV,BGM,FoodAndNutrition,Tea,Chaayos,1,139
20,2025-10-01,Delhi,TEAH35UPN3ZB4CPY,BGM,FoodAndNutrition,Tea,Chaayos,4,1396
18,2025-10-01,Mumbai,TEAH35UP8DYXN3HJ,BGM,FoodAndNutrition,Tea,Chaayos,3,417


### Prepping SKU MAPPING to MERGE with DUMP

In [607]:
sku_mapping_fkminutes = sku_mapping[["FKM_identifier","internal_id","internal_name","Category"]].copy()
sku_mapping_fkminutes.head()

,FKM_identifier,internal_id,internal_name,Category
0,TEAH35UP8DYXN3HJ,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
1,TEAHAYPBP7TMMYHG,5751.0,Chaayos Adrak Elaichi 250g,Chai Patti
2,<NA>,6330.0,Chaayos Adrak Elaichi 500g,Chai Patti
3,TEAH35UHKB2DCZPV,1504.0,Chaayos Masala Tea 100g,Chai Patti
4,TEAHAYPBUWYAKVQH,5749.0,Chaayos Masala Tea 250g,Chai Patti


In [608]:
fkminutes_base = df_fkminutes.merge(sku_mapping_fkminutes, on = "FKM_identifier", how = "left")
fkminutes_base.head()

,order_date_time,city,FKM_identifier,analytic_business_unit,analytic_super_category,analytic_vertical,brand_csv,units,mrp,internal_id,internal_name,Category
0,2025-10-01,Patna,TEAH35UHKB2DCZPV,BGM,FoodAndNutrition,Tea,Chaayos,1,139,1504.0,Chaayos Masala Tea 100g,Chai Patti
1,2025-10-01,Bangalore,TEAH35UP8DYXN3HJ,BGM,FoodAndNutrition,Tea,Chaayos,5,695,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
2,2025-10-01,Kolkata,TEAH35UHKB2DCZPV,BGM,FoodAndNutrition,Tea,Chaayos,1,139,1504.0,Chaayos Masala Tea 100g,Chai Patti
3,2025-10-01,Delhi,TEAH35UPN3ZB4CPY,BGM,FoodAndNutrition,Tea,Chaayos,4,1396,NaN,Discontinue,Discontinue
4,2025-10-01,Mumbai,TEAH35UP8DYXN3HJ,BGM,FoodAndNutrition,Tea,Chaayos,3,417,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti


## 5. Big Basket

In [609]:
df_bigbasket = pd.read_csv(f"{INPUT_DIR}/big-basket-raw.csv", low_memory=False)

df_bigbasket.drop(df_bigbasket.iloc[:, 0:10], axis=1, inplace=True)

for col in df_bigbasket.select_dtypes(include="object"):
    df_bigbasket[col] = df_bigbasket[col].str.strip().astype("string")

# Big Basket provides a date range string (YYYYMMDD) — split into start and end dates
df_bigbasket["start_date"] = pd.to_datetime(df_bigbasket["date_range"].str[:8],  format="%Y%m%d", errors="coerce")
df_bigbasket["end_date"]   = pd.to_datetime(df_bigbasket["date_range"].str[-8:], format="%Y%m%d", errors="coerce")

df_bigbasket = df_bigbasket[["start_date", "end_date", "date_range", "source_city_name", "brand_name",
                              "top_slug", "mid_slug", "leaf_slug", "source_sku_id", "sku_description",
                              "sku_weight", "total_quantity", "total_mrp", "total_sales"]]

In [610]:
df_bigbasket.rename(columns={"source_sku_id":"BB_identifier"}, inplace=True)

In [611]:
df_bigbasket.info()
df_bigbasket.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415 entries, 0 to 414
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   start_date        415 non-null    datetime64[ns]
 1   end_date          415 non-null    datetime64[ns]
 2   date_range        415 non-null    string        
 3   source_city_name  415 non-null    string        
 4   brand_name        415 non-null    string        
 5   top_slug          415 non-null    string        
 6   mid_slug          415 non-null    string        
 7   leaf_slug         415 non-null    string        
 8   BB_identifier     415 non-null    int64         
 9   sku_description   415 non-null    string        
 10  sku_weight        415 non-null    string        
 11  total_quantity    415 non-null    int64         
 12  total_mrp         415 non-null    int64         
 13  total_sales       415 non-null    float64       
dtypes: datetime64[ns](2), floa

,start_date,end_date,date_range,source_city_name,brand_name,top_slug,mid_slug,leaf_slug,BB_identifier,sku_description,sku_weight,total_quantity,total_mrp,total_sales
0,2025-10-01,2025-10-31,20251001 - 20251031,Bangalore,Chaayos,gourmet-world-food,drinks-beverages,gourmet-tea-tea-bags,40194562,Chaayos instant-desi-masala-chai-tea 25 pcs,25 pcs,61,11539,9480.20
1,2025-10-01,2025-10-31,20251001 - 20251031,Gurgaon,Chaayos,beverages,tea,leaf-dust-tea,40348267,Chaayos desi-masala-chai-tea 250 g,250 g,2,658,558.00
2,2025-10-01,2025-10-31,20251001 - 20251031,Mumbai,Chaayos,gourmet-world-food,drinks-beverages,gourmet-tea-tea-bags,40348270,Chaayos instant-cardamom-tea-premix 210 g (14...,210 g,8,1512,1329.11
3,2025-10-01,2025-10-31,20251001 - 20251031,Mumbai,Chaayos,beverages,energy-soft-drinks,icetea-non-aerated-drink,40348265,Chaayos instant-lemon-iced-tea-premix 220 g (...,220 g,27,3213,2588.47
4,2025-10-01,2025-10-31,20251001 - 20251031,Bangalore,Chaayos,gourmet-world-food,drinks-beverages,gourmet-tea-tea-bags,40348271,Chaayos detox-kahwa-herbal-green-tea 25 pcs,25 pcs,23,4807,3922.35


### Prepping SKU MAPPING to MERGE with DUMP

In [612]:
sku_mapping_bigbasket = sku_mapping[["BB_identifier","internal_id","internal_name","Category"]].copy()
sku_mapping_bigbasket.head()

,BB_identifier,internal_id,internal_name,Category
0,40194559.0,1505.0,Chaayos Adrak Elaichi 100g,Chai Patti
1,40348266.0,5751.0,Chaayos Adrak Elaichi 250g,Chai Patti
2,NaN,6330.0,Chaayos Adrak Elaichi 500g,Chai Patti
3,40194561.0,1504.0,Chaayos Masala Tea 100g,Chai Patti
4,40348267.0,5749.0,Chaayos Masala Tea 250g,Chai Patti


In [613]:
bigbasket_base = df_bigbasket.merge(sku_mapping_bigbasket, on = "BB_identifier", how = "left")
bigbasket_base.head()

,start_date,end_date,date_range,source_city_name,brand_name,top_slug,mid_slug,leaf_slug,BB_identifier,sku_description,sku_weight,total_quantity,total_mrp,total_sales,internal_id,internal_name,Category
0,2025-10-01,2025-10-31,20251001 - 20251031,Bangalore,Chaayos,gourmet-world-food,drinks-beverages,gourmet-tea-tea-bags,40194562,Chaayos instant-desi-masala-chai-tea 25 pcs,25 pcs,61,11539,9480.20,1960.0,Chaayos Masala Tea Bag,Tea Bag
1,2025-10-01,2025-10-31,20251001 - 20251031,Gurgaon,Chaayos,beverages,tea,leaf-dust-tea,40348267,Chaayos desi-masala-chai-tea 250 g,250 g,2,658,558.00,5749.0,Chaayos Masala Tea 250g,Chai Patti
2,2025-10-01,2025-10-31,20251001 - 20251031,Mumbai,Chaayos,gourmet-world-food,drinks-beverages,gourmet-tea-tea-bags,40348270,Chaayos instant-cardamom-tea-premix 210 g (14...,210 g,8,1512,1329.11,4973.0,Chaayos Cardamom Tea Premix Pack of 15,Premix
3,2025-10-01,2025-10-31,20251001 - 20251031,Mumbai,Chaayos,beverages,energy-soft-drinks,icetea-non-aerated-drink,40348265,Chaayos instant-lemon-iced-tea-premix 220 g (...,220 g,27,3213,2588.47,5916.0,Chaayos Lemon Iced Tea Premix,Ice Tea
4,2025-10-01,2025-10-31,20251001 - 20251031,Bangalore,Chaayos,gourmet-world-food,drinks-beverages,gourmet-tea-tea-bags,40348271,Chaayos detox-kahwa-herbal-green-tea 25 pcs,25 pcs,23,4807,3922.35,3672.0,Chaayos Detox Kahwa Tea Bag,Tea Bag


## EXPORTING BASE FILES

In [614]:
blinkit_base.to_csv(f"{OUTPUT_DIR}/blinkit-raw.csv", index = False)
zepto_base.to_csv(f"{OUTPUT_DIR}/zepto-raw.csv", index = False)
instamart_base.to_csv(f"{OUTPUT_DIR}/instamart-raw.csv", index = False)
fkminutes_base.to_csv(f"{OUTPUT_DIR}/fkminutes-raw.csv", index = False)
bigbasket_base.to_csv(f"{OUTPUT_DIR}/bigbasket-raw.csv", index = False)